# EBM Quantile Regression

This notebook covers quantile regression using pinball loss with Explainable Boosting Machines.

Standard regression models (e.g. with RMSE) predict the conditional mean of the target. Quantile regression instead predicts a specific quantile (e.g. median, 10th percentile, 90th percentile). This is useful for:

- **Prediction intervals**: Fit models at the 10th and 90th percentiles to get an 80% prediction interval.
- **Asymmetric risk**: When over-predicting and under-predicting have different costs.
- **Robustness**: Median regression (alpha=0.5) is more robust to outliers than mean regression.

EBMs support quantile regression via the `"quantile"` objective, which uses the pinball loss (also called the quantile loss). The pinball loss for quantile alpha is:

$$L(y, \hat{y}) = \begin{cases} \alpha \cdot (y - \hat{y}) & \text{if } y \geq \hat{y} \\ (1 - \alpha) \cdot (\hat{y} - y) & \text{if } y < \hat{y} \end{cases}$$

This loss penalizes under-predictions by a factor of alpha and over-predictions by a factor of (1 - alpha), causing the model to learn the alpha-quantile of the conditional distribution.

In [ ]:
# boilerplate
from interpret import show
from interpret.glassbox import ExplainableBoostingRegressor
import numpy as np

## Prediction Intervals

A key use case for quantile regression is constructing prediction intervals. By fitting separate models at different quantiles, we can estimate the range within which future observations are likely to fall.

Let's use a larger, noisier dataset to demonstrate this. We'll fit models at the 10th, 50th, and 90th percentiles to get an 80% prediction interval.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# Generate a single dataset and split it properly
# (using separate make_regression calls would create different underlying functions!)
X_all, y_all = make_regression(
    n_samples=1200, n_features=5, noise=20.0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=200, random_state=0)

# Fit quantile models at the 10th, 50th, and 90th percentiles
ebm_10 = ExplainableBoostingRegressor(objective="quantile:alpha=0.1")
ebm_50 = ExplainableBoostingRegressor(objective="quantile:alpha=0.5")
ebm_90 = ExplainableBoostingRegressor(objective="quantile:alpha=0.9")

ebm_10.fit(X_train, y_train)
ebm_50.fit(X_train, y_train)
ebm_90.fit(X_train, y_train)

pred_10 = ebm_10.predict(X_test)
pred_50 = ebm_50.predict(X_test)
pred_90 = ebm_90.predict(X_test)

print("First 5 test samples:")
print("  10th percentile:", np.round(pred_10[:5], 2))
print("  50th percentile:", np.round(pred_50[:5], 2))
print("  90th percentile:", np.round(pred_90[:5], 2))
print("  Actual y:       ", np.round(y_test[:5], 2))

In [ ]:
# Verify quantile ordering: q10 < q50 < q90 for most predictions
print("Fraction where q10 < q50:", np.mean(pred_10 < pred_50))
print("Fraction where q50 < q90:", np.mean(pred_50 < pred_90))

# Check empirical coverage of the 80% prediction interval [q10, q90]
coverage = np.mean((y_test >= pred_10) & (y_test <= pred_90))
print(f"Empirical coverage of [q10, q90] interval: {coverage:.1%} (target: ~80%)")

# Verify calibration: the fraction of actuals below each quantile prediction
# should approximately match the quantile level
frac_below_10 = np.mean(y_test < pred_10)
frac_below_50 = np.mean(y_test < pred_50)
frac_below_90 = np.mean(y_test < pred_90)
print(f"\nCalibration check (fraction of y_test below predictions):")
print(f"  q10 model: {frac_below_10:.1%} (target: ~10%)")
print(f"  q50 model: {frac_below_50:.1%} (target: ~50%)")
print(f"  q90 model: {frac_below_90:.1%} (target: ~90%)")

## Visualizing the Prediction Interval

Let's visualize the prediction interval on a sorted subset of test samples.

In [ ]:
try:
    import matplotlib
    import matplotlib.pyplot as plt

    # Sort by predicted median for a cleaner plot
    sort_idx = np.argsort(pred_50)
    x_axis = np.arange(len(sort_idx))

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(x_axis, pred_10[sort_idx], pred_90[sort_idx],
                    alpha=0.3, label="80% prediction interval (q10-q90)")
    ax.plot(x_axis, pred_50[sort_idx], label="Median prediction (q50)", linewidth=1.5)
    ax.scatter(x_axis, y_test[sort_idx], s=8, color="red", alpha=0.6, label="Actual values")
    ax.set_xlabel("Test samples (sorted by predicted median)")
    ax.set_ylabel("Target value")
    ax.set_title("EBM Quantile Regression: 80% Prediction Interval")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

## Calibration Across Quantiles

A well-calibrated quantile model at level alpha should have approximately alpha fraction of the true values falling below its predictions. Let's verify this across a range of quantile levels.

In [ ]:
alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
observed_fractions = []

for alpha in alphas:
    ebm_q = ExplainableBoostingRegressor(
        objective=f"quantile:alpha={alpha}", interactions=0)
    ebm_q.fit(X_train, y_train)
    preds_q = ebm_q.predict(X_test)
    observed_fractions.append(np.mean(y_test < preds_q))

try:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.scatter(alphas, observed_fractions, color="steelblue", s=60, zorder=5,
              label="EBM quantile model")
    ax.set_xlabel("Requested quantile (alpha)")
    ax.set_ylabel("Observed fraction below prediction")
    ax.set_title("Quantile Calibration on Test Set")
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()
except NameError:
    for a, f in zip(alphas, observed_fractions):
        print(f"  alpha={a:.2f}: observed={f:.2f}")

## Interpretability

One of the key advantages of quantile EBMs is that they remain fully interpretable. The global explanations show how each feature contributes to the predicted quantile, and local explanations show the additive score breakdown for individual predictions.

Let's compare the shape functions for the same feature across different quantiles.

In [ ]:
# Show global explanations for each quantile model
print("=== 10th Percentile Model ===")
show(ebm_10.explain_global())

In [ ]:
print("=== 90th Percentile Model ===")
show(ebm_90.explain_global())

In [ ]:
# Local explanation for a single test sample
show(ebm_50.explain_local(X_test[:5], y_test[:5]), 0)

## Summary

- Use `objective="quantile:alpha=0.5"` for median regression, or any alpha in (0, 1) for other quantiles.
- The prediction mechanism is identical to standard regression EBMs (intercept + additive score lookups). Only the training loss function changes.
- Fitting multiple quantile models (e.g. alpha=0.1 and alpha=0.9) provides interpretable prediction intervals.
- All EBM interpretability tools (global/local explanations) work with quantile models.